# Venice multimodal routing — experiment pipeline

This notebook is the **classical optimization experiment pipeline** for the Venice
delivery-routing project. It treats `VeniceTSPEnv` as the backend and compares:

- **Random feasible** policy
- **Nearest-neighbor** policy
- **Earliest-time-window** greedy policy
- **OR-Tools** with *no*, *soft*, and *strict* time windows (from `ORtools.py`)

It then loads an **already-trained DQN policy** (produced by `dqn_train.py`) and
runs a head-to-head comparison. The notebook never trains — it only runs
experiments and inference.

For every (method, seed) it records the full metric set, writes
`results/venice_experiment_results.csv`, and prints grouped summary tables.
Metrics are computed from the environment's own `simulate_route` transitions —
nothing is invented; anything that cannot be safely inferred is left as `None`.

## Imports and shared helpers

In [1]:
import csv
import dataclasses
import importlib.util
import os
import random
import time
import traceback
from dataclasses import dataclass
from typing import Any, Dict, List, Mapping, Optional, Sequence, Tuple

import numpy as np

from venice import Mode, VeniceTSPConfig, VeniceTSPEnv

# OR-Tools bridge (reuse the tested solver in ORtools.py; never reimplement it).
from ORtools import (
    build_multimodal_state_graph,
    build_required_node_matrices,
    expand_required_order,
    solve_with_ortools,
)

# DQN agent lives in its own module and depends on torch. Import it defensively
# so the classical pipeline still runs even if torch is unavailable here. Note:
# training is NOT done here -- see dqn_train.py. We only load a trained policy.
_dqn_import_error = None
try:
    from dqn_agent import DQNConfig, DQNAgent, evaluate_policy, greedy_route, observation_dim
    TORCH_AVAILABLE = True
except Exception as exc:  # torch (or its wheel for this Python) is missing
    TORCH_AVAILABLE = False
    _dqn_import_error = exc

ORTOOLS_AVAILABLE = importlib.util.find_spec("ortools") is not None
print(f"OR-Tools available: {ORTOOLS_AVAILABLE} | PyTorch available: {TORCH_AVAILABLE}")

OR-Tools available: True | PyTorch available: True


In [2]:
@dataclass
class ExperimentResult:
    """One (method, seed) evaluation. Unknown metrics are None, never faked."""

    method: str
    seed: int
    success: bool
    message: str = ""

    objective_cost: Optional[float] = None
    route_duration_minutes: Optional[float] = None
    travel_minutes: Optional[float] = None
    service_minutes: Optional[float] = None
    handling_minutes: Optional[float] = None
    monetary_cost: Optional[float] = None
    lateness_minutes: Optional[float] = None
    late_deliveries: Optional[int] = None
    bridge_crossings: Optional[int] = None
    mode_transfers: Optional[int] = None
    boat_legs: Optional[int] = None
    cart_legs: Optional[int] = None
    van_legs: Optional[int] = None
    delivered_count: Optional[int] = None
    num_deliveries: Optional[int] = None
    expanded_route_length: Optional[int] = None
    solve_time_seconds: Optional[float] = None


def make_env(num_deliveries: int, num_docks: int, topology_seed: int) -> VeniceTSPEnv:
    """Construct an environment instance with a deterministic topology."""
    config = VeniceTSPConfig(
        num_delivery_nodes=int(num_deliveries),
        num_docks=int(num_docks),
        seed=int(topology_seed),
    )
    return VeniceTSPEnv(config=config)


def _mean(values: Sequence[Optional[float]]) -> Optional[float]:
    nums = [float(v) for v in values if v is not None]
    return sum(nums) / len(nums) if nums else None


def metrics_from_simulation(env, sim, *, route, method, seed,
                            solve_time_seconds, message="") -> ExperimentResult:
    """Turn an env.simulate_route result into a fully populated record."""
    transitions = list(sim.get("transitions", []) or [])
    feasible = bool(sim.get("feasible", False))
    delivery_start = env.delivery_start

    if transitions:
        travel = sum(float(t["travel_time_minutes"]) for t in transitions)
        service = sum(float(t["service_time_minutes"]) for t in transitions)
        handling = sum(float(t["handling_time_minutes"]) for t in transitions)
        monetary = sum(float(t["monetary_cost"]) for t in transitions)
        lateness = sum(float(t["lateness_minutes"]) for t in transitions)
        late_deliveries = sum(
            1 for t in transitions
            if float(t["lateness_minutes"]) > 1e-9 and int(t["v"]) >= delivery_start
        )
        bridges = sum(int(t["bridges"]) for t in transitions)
        transfers = sum(1 for t in transitions if int(t["start_mode"]) != int(t["travel_mode"]))
        boat_legs = sum(1 for t in transitions if int(t["travel_mode"]) == int(Mode.BOAT))
        cart_legs = sum(1 for t in transitions if int(t["travel_mode"]) == int(Mode.CART))
        van_legs = sum(1 for t in transitions if int(t["travel_mode"]) == int(Mode.VAN))
        start_minutes = env.config.base_hour * 60.0
        end_minutes = float(sim.get("end_time_minutes", start_minutes))
        duration = end_minutes - start_minutes
    else:
        travel = service = handling = monetary = lateness = None
        late_deliveries = bridges = transfers = None
        boat_legs = cart_legs = van_legs = None
        duration = None

    objective = sim.get("objective_cost")
    return ExperimentResult(
        method=method, seed=int(seed), success=feasible,
        message=message or ("ok" if feasible else "; ".join(sim.get("violations", [])[:5])),
        objective_cost=None if objective is None else float(objective),
        route_duration_minutes=duration, travel_minutes=travel, service_minutes=service,
        handling_minutes=handling, monetary_cost=monetary, lateness_minutes=lateness,
        late_deliveries=late_deliveries, bridge_crossings=bridges, mode_transfers=transfers,
        boat_legs=boat_legs, cart_legs=cart_legs, van_legs=van_legs,
        delivered_count=int(sim.get("delivered_count", 0)),
        num_deliveries=int(env.num_delivery_nodes),
        expanded_route_length=int(len(route)),
        solve_time_seconds=None if solve_time_seconds is None else float(solve_time_seconds),
    )


def failed_result(method, seed, message, solve_time_seconds=None) -> ExperimentResult:
    """A clean record for a method that could not produce a route at all."""
    return ExperimentResult(method=method, seed=int(seed), success=False,
                            message=message, solve_time_seconds=solve_time_seconds)

## Baseline route builders

Every baseline produces a concrete route that is then scored by `env.simulate_route`, so all methods share one honest metric path.

In [3]:
def random_feasible_rollout(env, rng) -> List[int]:
    """Walk the live env choosing uniformly among feasible actions."""
    env.reset(seed=rng.randint(0, 2**31 - 1))
    route = []
    for _ in range(env.max_episode_steps):
        feasible = env.feasible_actions()
        if not feasible:
            break
        action = rng.choice(feasible)
        _, _, terminated, truncated, _ = env.step(action)
        route.append(int(action))
        if terminated or truncated:
            break
    return route


def _ordered_route_via_legs(order_indices, required_nodes, leg_paths) -> List[int]:
    """Expand an order over required-node indices into an executable route."""
    order_nodes = [int(required_nodes[i]) for i in order_indices]
    expanded, _legs = expand_required_order(order_nodes, required_nodes, leg_paths)
    return list(expanded)


def nearest_neighbor_order(cost_matrix) -> List[int]:
    """Classic nearest-neighbor tour over required-node indices (depot 0 start/end)."""
    n = cost_matrix.shape[0]
    unvisited = set(range(1, n))
    order, current = [0], 0
    while unvisited:
        nxt = min(unvisited, key=lambda j: cost_matrix[current, j])
        order.append(nxt)
        unvisited.remove(nxt)
        current = nxt
    order.append(0)
    return order


def earliest_time_window_order(env, required_nodes) -> List[int]:
    """Earliest-due-date ordering of deliveries (depot first and last)."""
    index_of = {int(node): i for i, node in enumerate(required_nodes)}
    deliveries = [int(n) for n in required_nodes[1:]]
    deliveries.sort(key=lambda n: (env.nodes[n].time_window_end,
                                   env.nodes[n].time_window_start, n))
    return [0] + [index_of[n] for n in deliveries] + [0]


def run_baseline_method(env, method, seed, rng) -> ExperimentResult:
    """Run one non-OR-Tools baseline and return its metric record."""
    require_return = env.config.require_return_to_depot
    start = time.perf_counter()

    if method == "random":
        route = random_feasible_rollout(env, rng)
    elif method in ("nearest_neighbor", "earliest_tw"):
        env.reset(seed=seed)
        graph = build_multimodal_state_graph(env)
        delivery_nodes = [int(x) for x in env.delivery_nodes.tolist()]
        required_nodes = [int(env.depot_node)] + delivery_nodes
        cost_matrix, _t, leg_paths, unreachable = build_required_node_matrices(
            env, graph, required_nodes)
        if unreachable:
            return failed_result(method, seed, f"Unreachable required pairs: {unreachable[:5]}",
                                 solve_time_seconds=time.perf_counter() - start)
        order = (nearest_neighbor_order(cost_matrix) if method == "nearest_neighbor"
                 else earliest_time_window_order(env, required_nodes))
        try:
            route = _ordered_route_via_legs(order, required_nodes, leg_paths)
        except KeyError as exc:
            return failed_result(method, seed, f"Leg expansion failed for pair {exc}",
                                 solve_time_seconds=time.perf_counter() - start)
    else:
        raise ValueError(f"Unknown baseline method: {method}")

    solve_time = time.perf_counter() - start
    sim = env.simulate_route(
        route, start_node=env.depot_node, start_mode=Mode.VAN,
        start_time_minutes=env.config.base_hour * 60.0,
        require_all_deliveries=True, require_return_to_depot=require_return,
        penalize_revisits=True,
    )
    return metrics_from_simulation(env, sim, route=route, method=method, seed=seed,
                                   solve_time_seconds=solve_time)

## OR-Tools method

Wraps `solve_with_ortools` for the `none` / `soft` / `strict` time-window regimes, and reports the method as unavailable if OR-Tools is not installed.

In [4]:
def run_ortools_method(env, time_window_mode, seed, time_limit_seconds) -> ExperimentResult:
    """Run one OR-Tools time-window regime, or report it unavailable."""
    method = f"ortools_{time_window_mode}"
    if not ORTOOLS_AVAILABLE:
        return failed_result(method, seed, "OR-Tools not installed (method unavailable)")

    env.reset(seed=seed)
    start = time.perf_counter()
    try:
        result = solve_with_ortools(env, time_window_mode=time_window_mode,
                                    time_limit_seconds=int(time_limit_seconds), verbose=False)
    except Exception as exc:
        return failed_result(method, seed, f"OR-Tools error: {exc}",
                             solve_time_seconds=time.perf_counter() - start)

    solve_time = time.perf_counter() - start
    sim = dict(result.simulated)
    return metrics_from_simulation(env, sim, route=result.expanded_route, method=method,
                                   seed=seed, solve_time_seconds=solve_time,
                                   message=result.message)

## Experiment runner, summary, and CSV export

In [5]:
BASELINE_METHODS = ("random", "nearest_neighbor", "earliest_tw")
ORTOOLS_MODES = ("none", "soft", "strict")


def run_single_experiment(method, seed, *, num_deliveries, num_docks,
                          time_limit_seconds) -> ExperimentResult:
    """Evaluate one method on one seed; any exception becomes a failed record."""
    try:
        env = make_env(num_deliveries, num_docks, topology_seed=seed)
        if method in BASELINE_METHODS:
            return run_baseline_method(env, method, seed, random.Random(seed))
        if method.startswith("ortools_"):
            return run_ortools_method(env, method.split("_", 1)[1], seed, time_limit_seconds)
        raise ValueError(f"Unknown method: {method}")
    except Exception as exc:
        return failed_result(method, seed, f"Unhandled error: {exc}\n{traceback.format_exc(limit=2)}")


def run_all_experiments(*, num_seeds, num_deliveries, num_docks, base_seed,
                        time_limit_seconds, methods=None, verbose=True) -> List[ExperimentResult]:
    """Run every method across num_seeds deterministic instances."""
    if methods is None:
        methods = list(BASELINE_METHODS) + [f"ortools_{m}" for m in ORTOOLS_MODES]
    results = []
    for s in (base_seed + i for i in range(num_seeds)):
        for method in methods:
            result = run_single_experiment(method, s, num_deliveries=num_deliveries,
                                           num_docks=num_docks, time_limit_seconds=time_limit_seconds)
            results.append(result)
            if verbose:
                status = "ok " if result.success else "x  "
                cost = "n/a" if result.objective_cost is None else f"{result.objective_cost:9.1f}"
                tail = "" if result.success else result.message[:60]
                print(f"  [{status}] seed={s:<5} {method:<16} cost={cost}  {tail}")
    return results


def _fmt(value, width, prec=2):
    return f"{'n/a':>{width}}" if value is None else f"{value:>{width}.{prec}f}"


def _print_summary_table(summary):
    header = (f"{'method':<18}{'runs':>6}{'success':>9}{'obj_cost':>12}"
              f"{'lateness':>11}{'duration':>11}{'solve_s':>10}")
    print("\n" + "=" * len(header))
    print("Experiment summary (means over successful runs; success rate over all)")
    print("=" * len(header))
    print(header)
    print("-" * len(header))
    for method, row in summary.items():
        sr = row["success_rate"]
        sr_str = "n/a" if sr is None else f"{sr * 100:6.1f}%"
        print(f"{method:<18}{int(row['runs'] or 0):>6}{sr_str:>9}"
              f"{_fmt(row['mean_objective_cost'], 12, 1)}"
              f"{_fmt(row['mean_lateness_minutes'], 11, 1)}"
              f"{_fmt(row['mean_route_duration_minutes'], 11, 1)}"
              f"{_fmt(row['mean_solve_time_seconds'], 10, 3)}")
    print("=" * len(header) + "\n")


def summarize_results(results):
    """Group records by method, print a clean table, and return the summary dict."""
    methods = sorted({r.method for r in results})
    summary = {}
    for method in methods:
        rows = [r for r in results if r.method == method]
        successful = [r for r in rows if r.success]
        n = len(rows)
        summary[method] = {
            "runs": float(n),
            "success_rate": (len(successful) / n) if n else None,
            "mean_objective_cost": _mean([r.objective_cost for r in successful]),
            "mean_lateness_minutes": _mean([r.lateness_minutes for r in successful]),
            "mean_route_duration_minutes": _mean([r.route_duration_minutes for r in successful]),
            "mean_solve_time_seconds": _mean([r.solve_time_seconds for r in rows]),
        }
    _print_summary_table(summary)
    return summary


def save_results_csv(results, path):
    """Write all records to CSV, creating the parent directory if needed."""
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
    fieldnames = [f.name for f in dataclasses.fields(ExperimentResult)]
    with open(path, "w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        for result in results:
            writer.writerow(dataclasses.asdict(result))
    print(f"Saved {len(results)} records to {path}")
    return path

## Part A — run the classical experiments

Adjust the parameters below, then run. Results are written to `results/venice_experiment_results.csv`.

In [6]:
# --- experiment parameters ---
NUM_SEEDS = 20
NUM_DELIVERIES = 15
NUM_DOCKS = 6
BASE_SEED = 7
TIME_LIMIT_SECONDS = 10
OUTPUT_DIR = "."

results = run_all_experiments(
    num_seeds=NUM_SEEDS,
    num_deliveries=NUM_DELIVERIES,
    num_docks=NUM_DOCKS,
    base_seed=BASE_SEED,
    time_limit_seconds=TIME_LIMIT_SECONDS,
)

csv_path = os.path.join(OUTPUT_DIR, "results", "venice_experiment_results.csv")
save_results_csv(results, csv_path)
summary = summarize_results(results)

  [ok ] seed=7     random           cost=  13784.2  
  [ok ] seed=7     nearest_neighbor cost=   8681.4  
  [ok ] seed=7     earliest_tw      cost=    750.9  


  [ok ] seed=7     ortools_none     cost=   8493.0  


  [ok ] seed=7     ortools_soft     cost=    780.1  
  [ok ] seed=7     ortools_strict   cost=    750.9  
  [ok ] seed=8     random           cost=  22098.2  
  [ok ] seed=8     nearest_neighbor cost=   6514.8  
  [ok ] seed=8     earliest_tw      cost=   4118.5  


  [ok ] seed=8     ortools_none     cost=   5366.7  


  [ok ] seed=8     ortools_soft     cost=    783.4  
  [ok ] seed=8     ortools_strict   cost=   4118.5  
  [ok ] seed=9     random           cost=  26951.7  
  [ok ] seed=9     nearest_neighbor cost=   9373.4  
  [ok ] seed=9     earliest_tw      cost=   2507.9  


...

  [ok ] seed=24    ortools_soft     cost=    656.4  
  [ok ] seed=24    ortools_strict   cost=   2113.7  
  [ok ] seed=25    random           cost=  20186.5  
  [ok ] seed=25    nearest_neighbor cost=   9079.8  
  [ok ] seed=25    earliest_tw      cost=   2343.5  


  [ok ] seed=25    ortools_none     cost=   6341.8  


  [ok ] seed=25    ortools_soft     cost=    822.0  
  [ok ] seed=25    ortools_strict   cost=   2343.5  
  [ok ] seed=26    random           cost=  22400.4  
  [ok ] seed=26    nearest_neighbor cost=   4867.8  
  [ok ] seed=26    earliest_tw      cost=    762.4  


  [ok ] seed=26    ortools_none     cost=   9043.1  


  [ok ] seed=26    ortools_soft     cost=    838.3  
  [ok ] seed=26    ortools_strict   cost=    762.4  
Saved 120 records to ./results/venice_experiment_results.csv

Experiment summary (means over successful runs; success rate over all)
method              runs  success    obj_cost   lateness   duration   solve_s
-----------------------------------------------------------------------------
earliest_tw           20   100.0%      1555.1      160.5      484.3     0.008
nearest_neighbor      20   100.0%      8346.4     1510.6      511.5     0.008
ortools_none          20   100.0%      8066.0     1465.5      474.9    10.016
ortools_soft          20   100.0%       777.1       30.8      398.2    10.010
ortools_strict        20   100.0%      1555.1      160.5      484.3     0.034
random                20   100.0%     23978.7     4188.8      955.6     0.011



# Part B - Train the DQN

The DQN policy is trained **separately** by `dqn_train.py`; this notebook only
loads the saved `final_model.pt` and runs it. To (re)train:

```
python dqn_train.py --episodes 2000 --deliveries 15 --docks 6 --seed 7
```

## Part C — load the trained DQN and compare

The checkpoint is self-describing (it stores the instance dimensions), so the
agent is rebuilt on the exact environment it was trained on.

In [7]:
MODEL_PATH = os.path.join(OUTPUT_DIR, "runs", "dqn_venice", "final_model.pt")

agent = None
dqn_env = None
if not TORCH_AVAILABLE:
    print(f"[DQN skipped] dqn_agent could not be imported: {_dqn_import_error}\n"
          f"Install PyTorch to load/run the trained policy.")
elif not os.path.exists(MODEL_PATH):
    print(f"[DQN skipped] No trained model at {MODEL_PATH}.\n"
          f"Train one first:  python dqn_train.py --deliveries {NUM_DELIVERIES} "
          f"--docks {NUM_DOCKS} --seed 7")
else:
    import torch
    ckpt = torch.load(MODEL_PATH, map_location="cpu", weights_only=False)
    cfg_fields = {f.name for f in dataclasses.fields(DQNConfig)}
    cfg = DQNConfig(**{k: v for k, v in ckpt["config"].items() if k in cfg_fields})
    dqn_env = make_env(cfg.num_deliveries, cfg.num_docks, topology_seed=cfg.seed)
    agent = DQNAgent(observation_dim(dqn_env), dqn_env.total_nodes, cfg)
    agent.load(MODEL_PATH)
    print(f"Loaded trained DQN from {MODEL_PATH} "
          f"(deliveries={cfg.num_deliveries}, docks={cfg.num_docks}, seed={cfg.seed})")

    eval_metrics = evaluate_policy(dqn_env, agent, episodes=20)
    print("\nGreedy evaluation of the loaded DQN:")
    for key, value in eval_metrics.items():
        print(f"  {key:<24} {value}")

[DQN skipped] No trained model at ./runs/dqn_venice/final_model.pt.
Train one first:  python dqn_train.py --deliveries 15 --docks 6 --seed 7


In [8]:
def compare_policies(*, num_deliveries, num_docks, base_seed, time_limit_seconds,
                     agent, ortools_mode="soft", eval_seeds=20):
    """Score random / nearest-neighbor / DQN / OR-Tools on the same instances."""
    methods = ["random", "nearest_neighbor"]
    if agent is not None:
        methods.append("dqn")
    if ORTOOLS_AVAILABLE:
        methods.append(f"ortools_{ortools_mode}")

    all_results = []
    for i in range(eval_seeds):
        seed = base_seed + i
        env = make_env(num_deliveries, num_docks, topology_seed=seed)
        for method in methods:
            if method == "dqn":
                route = greedy_route(env, agent, seed)
                sim = env.simulate_route(
                    route, start_node=env.depot_node, start_mode=Mode.VAN,
                    start_time_minutes=env.config.base_hour * 60.0,
                    require_all_deliveries=True,
                    require_return_to_depot=env.config.require_return_to_depot,
                    penalize_revisits=True,
                )
                all_results.append(metrics_from_simulation(
                    env, sim, route=route, method="dqn", seed=seed, solve_time_seconds=None))
            elif method.startswith("ortools_"):
                all_results.append(run_ortools_method(env, ortools_mode, seed, time_limit_seconds))
            else:
                all_results.append(run_baseline_method(env, method, seed, random.Random(seed)))

    print("\nPolicy comparison (random / nearest-neighbor / DQN / OR-Tools):")
    return summarize_results(all_results)


# Use the DQN's own training dimensions so its observation/action sizes match.
cmp_deliveries = agent.config.num_deliveries if agent is not None else NUM_DELIVERIES
cmp_docks = agent.config.num_docks if agent is not None else NUM_DOCKS

comparison = compare_policies(
    num_deliveries=cmp_deliveries, num_docks=cmp_docks, base_seed=BASE_SEED,
    time_limit_seconds=TIME_LIMIT_SECONDS, agent=agent, ortools_mode="soft",
)


Policy comparison (random / nearest-neighbor / DQN / OR-Tools):

Experiment summary (means over successful runs; success rate over all)
method              runs  success    obj_cost   lateness   duration   solve_s
-----------------------------------------------------------------------------
nearest_neighbor      20   100.0%      8346.4     1510.6      511.5     0.008
ortools_soft          20   100.0%       777.1       30.8      398.2    10.010
random                20   100.0%     23978.7     4188.8      955.6     0.011



## Part C.1 — DQN outcome: an honest negative result

The reinforcement-learning approach (`dqn_agent.py` / `dqn_train.py`) was trained
on GPU via `modal_train.py` (Double-DQN with strict action masking, Weights &
Biases logging). **It did not converge into a usable policy.**

Across a 26,000-episode run (cancelled before the full 40k):

- Greedy evaluation **success rate stayed at 0%** at every checkpoint — the policy
  never completed a full feasible tour (all 15 deliveries + return to depot) on
  held-out instances.
- Training episodes increasingly hit the step cap (`len=138`), i.e. routes timed
  out instead of finishing.
- Even after epsilon fully decayed (`eps=0.05`), the greedy policy could not
  string together a complete delivery sequence.

**Why this is hard for vanilla DQN here:**

1. **Sparse, long-horizon success.** A reward only "pays off" after ~40–130 correct
   sequential decisions ending back at the depot; random exploration almost never
   stumbles onto a complete feasible tour to learn from.
2. **Missing deadline information.** The observation encodes the current node, mode,
   delivered mask, clock time, and flood mask — but **not each delivery's time
   window**. The agent has to infer deadlines from experience, which is exactly the
   signal OR-Tools uses explicitly.
3. **Combinatorial action space.** Choosing the next node out of all nodes, every
   step, is a large branching problem for a value-based method without search,
   curriculum, or reward shaping.

We keep the DQN in the repo as a documented attempt. The next section pivots to a
**simple supervised model** that sidesteps these issues by *learning from* the
classical solver instead of discovering routes from scratch.


## Part D — Supervised delivery-order model (learning to imitate OR-Tools)

Instead of learning to route from scratch (RL), we train a **simple supervised
model** to imitate the OR-Tools `soft` solver — a learning-to-rank / learned
dispatch policy (`ml_ranker.py`):

1. **Labels.** Solve a set of *training* instances (seeds disjoint from the eval
   set) with OR-Tools soft time windows to get high-quality delivery orders.
2. **Decision-step features.** Replay each solved order. At every step, build a
   feature vector for each still-undelivered delivery — leg cost & time from the
   current node, the delivery's time window, the implied arrival / slack / wait /
   lateness, demand, flood flag, and fraction of deliveries remaining. The label
   is which delivery the solver visited next.
3. **Model.** A small MLP (PyTorch, CPU) scores candidates; trained as a pointwise
   ranker (binary "is this the next stop?" with class weighting).
4. **Inference.** From the depot, greedily pick the highest-scored undelivered
   delivery, advance an approximate clock, repeat. The resulting order is expanded
   into an executable multimodal route with the **same** `expand_required_order`
   machinery the OR-Tools bridge uses, then scored by `env.simulate_route` — the
   identical metric path used by every other method here.

This trains in ~2–4 minutes on CPU (the cost is OR-Tools label generation, not the
model itself), and — unlike the DQN — produces feasible, high-quality routes.


In [9]:
from ml_ranker import RankerConfig, train_ranker, ranker_route, save_ranker

# Train on instances whose seeds are disjoint from the evaluation seeds
# (BASE_SEED .. BASE_SEED + NUM_SEEDS) so results reflect generalization.
ranker_config = RankerConfig(
    num_deliveries=NUM_DELIVERIES,
    num_docks=NUM_DOCKS,
    epochs=60,
    ortools_time_limit_seconds=5,
    train_seeds=tuple(range(100, 130)),  # 30 held-out training instances
    seed=0,
)

print("Generating OR-Tools labels and training the supervised ranker...")
ranker_model, ranker_info = train_ranker(ranker_config, verbose=True)

ranker_path = os.path.join(OUTPUT_DIR, "runs", "ml_ranker", "ranker.pt")
save_ranker(ranker_model, ranker_config, ranker_path)
print(
    f"\nTrained on {ranker_info['num_rows']} decision rows "
    f"(final loss {ranker_info['final_loss']:.4f}). Saved to {ranker_path}"
)


Generating OR-Tools labels and training the supervised ranker...


Built dataset from 30/30 solved instances: 3600 rows, 450 positives.


  epoch   1/60  loss=1.1225
  epoch   7/60  loss=0.8007
  epoch  13/60  loss=0.7329
  epoch  19/60  loss=0.6655
  epoch  25/60  loss=0.6793


  epoch  31/60  loss=0.6706
  epoch  37/60  loss=0.6292
  epoch  43/60  loss=0.6120
  epoch  49/60  loss=0.6096
  epoch  55/60  loss=0.6558
  epoch  60/60  loss=0.6442

Trained on 3600 decision rows (final loss 0.6442). Saved to ./runs/ml_ranker/ranker.pt


In [10]:
def compare_with_ranker(*, num_deliveries, num_docks, base_seed, time_limit_seconds,
                         ranker_model, dqn_agent=None, eval_seeds=20):
    """Score every method on identical instances, including the supervised ranker."""

    methods = ["random", "nearest_neighbor", "earliest_tw", "ml_ranker"]
    if dqn_agent is not None:
        methods.append("dqn")
    if ORTOOLS_AVAILABLE:
        methods.append("ortools_soft")

    all_results = []
    for i in range(eval_seeds):
        seed = base_seed + i
        env = make_env(num_deliveries, num_docks, topology_seed=seed)
        for method in methods:
            if method == "ml_ranker":
                start = time.perf_counter()
                env.reset(seed=seed)
                route = ranker_route(env, ranker_model)
                solve_time = time.perf_counter() - start
                if not route:
                    all_results.append(failed_result(method, seed, "Unreachable required pairs",
                                                      solve_time_seconds=solve_time))
                    continue
                sim = env.simulate_route(
                    route, start_node=env.depot_node, start_mode=Mode.VAN,
                    start_time_minutes=env.config.base_hour * 60.0,
                    require_all_deliveries=True,
                    require_return_to_depot=env.config.require_return_to_depot,
                    penalize_revisits=True,
                )
                all_results.append(metrics_from_simulation(
                    env, sim, route=route, method=method, seed=seed, solve_time_seconds=solve_time))
            elif method == "dqn":
                route = greedy_route(env, dqn_agent, seed)
                sim = env.simulate_route(
                    route, start_node=env.depot_node, start_mode=Mode.VAN,
                    start_time_minutes=env.config.base_hour * 60.0,
                    require_all_deliveries=True,
                    require_return_to_depot=env.config.require_return_to_depot,
                    penalize_revisits=True,
                )
                all_results.append(metrics_from_simulation(
                    env, sim, route=route, method="dqn", seed=seed, solve_time_seconds=None))
            elif method.startswith("ortools_"):
                all_results.append(run_ortools_method(env, method.split("_", 1)[1], seed, time_limit_seconds))
            else:
                all_results.append(run_baseline_method(env, method, seed, random.Random(seed)))

    print("\nFinal comparison (baselines / supervised ranker / OR-Tools):")
    summary = summarize_results(all_results)
    save_results_csv(all_results, os.path.join(OUTPUT_DIR, "results", "venice_with_ranker_results.csv"))
    return summary


ranker_comparison = compare_with_ranker(
    num_deliveries=NUM_DELIVERIES, num_docks=NUM_DOCKS, base_seed=BASE_SEED,
    time_limit_seconds=TIME_LIMIT_SECONDS, ranker_model=ranker_model,
    dqn_agent=agent, eval_seeds=NUM_SEEDS,
)



Final comparison (baselines / supervised ranker / OR-Tools):

Experiment summary (means over successful runs; success rate over all)
method              runs  success    obj_cost   lateness   duration   solve_s
-----------------------------------------------------------------------------
earliest_tw           20   100.0%      1555.1      160.5      484.3     0.008
ml_ranker             20   100.0%       893.1       44.4      430.1     0.009
nearest_neighbor      20   100.0%      8346.4     1510.6      511.5     0.008
ortools_soft          20   100.0%       777.1       30.8      398.2    10.009
random                20   100.0%     23978.7     4188.8      955.6     0.011

Saved 100 records to ./results/venice_with_ranker_results.csv
